# Dynamic GNNs on Wikipedia Dataset  
**Архитектуры для сравнения**  
- GCN (static baseline)  
- EvolveGCN-O (snapshot-based dynamic model)  
- TGAT (temporal graph attention model)

**Датасет:** Wikipedia edits (JODIE)  
Each event: (user → page, timestamp)




In [ ]:
# Если выполняете в Google Colab, установите PyTorch Geometric:
!pip install torch_geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv
from sklearn.metrics import roc_auc_score



## Load and preprocess the Wikipedia dynamic graph dataset
Dataset: https://snap.stanford.edu/jodie/wikipedia.csv  
Each event:  
`user_id, item_id, timestamp`


In [ ]:
url = "https://snap.stanford.edu/jodie/wikipedia.csv"
df = pd.read_csv(url, header=None, usecols=[0,1,2])
df.columns = ["user","item","timestamp"]

# clean timestamps
df["timestamp"] = pd.to_numeric(df["timestamp"].astype(str).str.strip(), errors="coerce")
df = df.dropna(subset=["timestamp"])
df["timestamp"] = df["timestamp"].astype(int)

df = df.sort_values("timestamp").reset_index(drop=True)

print(df.shape)
df.head()


/tmp/ipython-input-728066216.py:2: DtypeWarning: Columns (0,1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url, header=None, usecols=[0,1,2])


(157474, 3)


,user,item,timestamp
0,0,0,0
1,1,1,36
2,1,1,77
3,2,2,131
4,1,1,150


## Reindex user and item IDs  
We map users → [0..U-1], items → [U..U+I-1]


In [ ]:
# Извлекаем уникальные ID всех пользователей
users = df.user.unique()

# Извлекаем уникальные ID всех items (страниц Wikipedia)
items = df.item.unique()

# Создаём отображение "старый user_id → новый индекс"
u_map = {u: i for i, u in enumerate(users)}

# Создаём отображение "старый item_id → новый индекс"
# items идут сразу после users, то есть item0 получает индекс num_users,
# item1 → num_users+1, и так далее.
# Это гарантирует, что все узлы лежат в одном общем node space
i_map = {it: i + len(users) for i, it in enumerate(items)}

# Применяем отображения:
# user_id → src (узел в первой доле)
df["src"] = df.user.map(u_map)

# item_id → dst (узел во второй доле)
df["dst"] = df.item.map(i_map)

# Общее число узлов = число пользователей + число items
# Эти узлы образуют двудольный граф User–Item
nodes = len(users) + len(items)


## Snapshot construction  
Snapshots = 1 day (Δt = 86400 seconds)


In [ ]:
# TODO


31

## Negative sampling  
We sample (user, item) pairs that do not exist in data.


In [ ]:
def neg_sample(pos_df, num_users, num_items):
    # TODO

    # Возвращаем тензор размером [num_pos, 2]
    return torch.tensor(neg)



## Split into train/val/test snapshots  



In [ ]:
train_s = snapshots[:4]
val_s   = snapshots[4:5]
test_s  = snapshots[5:]

train_df = pd.concat(train_s)
val_df   = pd.concat(val_s)
test_df  = pd.concat(test_s)

print(train_df.shape, val_df.shape, test_df.shape)


(16463, 6) (4512, 6) (136499, 6)


## Convert pandas edges → PyG edge_index


Преобразование табличного списка рёбер (pandas DataFrame)
в формат edge_index, совместимый с PyTorch Geometric.

df уже содержит два столбца:
- df.src — индекс узла-источника (user)
- df.dst — индекс узла-получателя (item)

Мы преобразуем их в edge_index формата [2, num_edges],
где первая строка — список источников, вторая — список целевых узлов.

In [ ]:
def to_ei(df):
    # Преобразуем столбец src (users) в тензор целых чисел
    src = torch.tensor(df.src.values, dtype=torch.long)

    # Преобразуем столбец dst (items) в тензор
    dst = torch.tensor(df.dst.values, dtype=torch.long)

    # Мы строим неориентированное ребро.
    #
    # PyG ожидает, что если граф неориентирован, то каждое ребро
    # должно быть задано в обоих направлениях:
    #   u → v и v → u.
    #
    # Поэтому создаём:
    #   [src, dst]  — прямые направления
    #   [dst, src]  — обратные направления
    #
    # torch.cat соединяет их вдоль оси 0.
    # Далее stack формирует матрицу:
    #   edge_index =
    #     [[u1, u2, ..., v1, v2, ...],
    #      [v1, v2, ..., u1, u2, ...]]
    #
    # Такая форма требуется GCNConv и другим PyG-операторам.
    return torch.stack([
        torch.cat([src, dst]),  # первая строка — источники всех рёбер
        torch.cat([dst, src])   # вторая строка — цели всех рёбер
    ])



# Models

## Static Baseline: Модель GCN

Базовая двухслойная Graph Convolutional Network:

- принимает матрицу признаков узлов `X`
- использует `GCNConv`, где обновление происходит по формуле  
  $ H = \hat{A} X W $, где  
  $ \hat{A} = D^{-1/2} A D^{-1/2} $
- первый слой уменьшает или увеличивает размерность до скрытой `hid`
- второй слой формирует финальные эмбеддинги размерности `out_dim`

Используется как статический baseline в задаче линк-предсказания.



In [ ]:
class GCN(nn.Module):
    def __init__(self, in_dim, hid, out_dim):
        super().__init__()
        # TODO
        return self.c2(h, ei)


### Training loop

### Оценка качества: score и AUC

**score(u, v)**  
Мы используем скалярное произведение эмбеддингов узлов \( h_u \) и \( h_v \):  
если узлы похожи — score высокий, если нет — низкий.

**neg_sample**  
Для каждого позитивного ребра (u, v) генерируется одно отрицательное  
(u, v'), которого нет в графе.

**eval_auc**  
AUC-ROC показывает, насколько модель успешно различает реальные рёбра  
(метка 1) от случайных рёбер (метка 0).

AUC = 0.5 → случайное угадывание  
AUC → 1.0 → идеальное различение


In [ ]:
def score(u, v, emb):
    # TODO
    return res_score


def eval_auc(emb, pos_df):
    # TODO
    return roc_auc_score(labels, scores)



### Построение статического графа и GCN baseline

Для сравнения с динамическими моделями мы обучаем обычный двухслойный GCN:

- строим статичный граф из всех обучающих событий,
- используем one-hot признаки узлов `X = I`,
- обучаем GCN предсказывать вероятность существования ребра (link prediction).

Это baseline, не учитывающий время.  
Он помогает понять, насколько полезна динамическая архитектура (например TGAT или EvolveGCN).


In [ ]:
train_ei = to_ei(train_df)
x = torch.eye(nodes) # сделали one-hot признаки узлов

gcn = GCN(nodes, 64, 64)
opt = torch.optim.Adam(gcn.parameters(), lr=1e-3)


### Обучение статического GCN (link prediction)

На каждой эпохе:

1. **Строим эмбеддинги узлов** через GCN  
2. **Извлекаем positive edges** из train-данных  
3. **Генерируем negative edges** через негативное сэмплирование  
4. **Вычисляем score(u, v)** как скалярное произведение эмбеддингов  
5. Формируем loss:
   - реальные рёбра должны иметь высокую вероятность  
   - отрицательные — низкую  
6. Обновляем параметры GCN  
7. Вычисляем AUC на валидации  
8. Печатаем лог эпохи  

AUC показывает, насколько хорошо модель различает существующие и несуществующие связи.


In [ ]:
for ep in range(3):   # цикл по эпохам обучения
    # 1. Прямое распространение через GCN
    emb = gcn(x, train_ei)

    # 2. Формирование позитивных примеров
    pos = torch.tensor(train_df[['src', 'dst']].values)

    # 3. Сэмплирование негативных примеров
    neg = neg_sample(train_df, len(users), len(items))

    # 4. Счёт позитивных скорингов
    ps = torch.stack([score(u, v, emb) for u, v in pos])

    # 5. Счёт негативных скорингов
    ns = torch.stack([score(u, v, emb) for u, v in neg])

    # 6. Функция потерь
    loss = -(torch.log(torch.sigmoid(ps)).mean() +
             torch.log(1 - torch.sigmoid(ns)).mean())

    # 7. Обратное распространение (Backprop)
    opt.zero_grad()    # сбрасываем градиенты
    loss.backward()    # считаем новые
    opt.step()         # обновляем веса GCN

    #  8. Оценка качества
    # Считаем AUC на валидационных данных:
    val_auc = eval_auc(emb, val_df)

    # 9. Лог обучения
    print(f"GCN Epoch {ep}: loss={loss.item():.4f}, val AUC={val_auc:.4f}")


## Архитектура EvolveGCN-O

EvolveGCN-O — это динамическая GNN, которая обновляет весовые матрицы GCN во времени с помощью рекуррентной сети (GRU).  
На каждом шаге времени $t$ модель получает snapshot-граф $G_t$ и генерирует новые веса $W_t$ для GCN.



###  Основная идея
Вместо того чтобы учить GCN с фиксированными весами,  
EvolveGCN-O делает веса динамическими, то есть:

$$
W_{t+1} = \mathrm{GRU}( \, \mathrm{vec}(W_t), \, \mathrm{vec}(W_t) \,)
$$

где  
- $W_t$ — матрица весов GCN на шаге $t$,  
- $\mathrm{vec}(W_t)$ — векторизация матрицы (разворачивание в 1D).

Таким образом, веса эволюционируют вместе с графом.

---

###  Работа на каждом шаге времени

#### 1. Обновление весовой матрицы через GRU

Матрица $W_t$ размером $(d_{\text{out}} \times d_{\text{in}})$ разворачивается в вектор.  
GRU обновляет её:

$$
\mathbf{w}_t = \mathrm{vec}(W_t)
$$

$$
\mathbf{w}_{t+1} = \mathrm{GRUCell}(\mathbf{w}_t, \mathbf{w}_t)
$$

После чего она снова превращается в матрицу:

$$
W_{t+1} = \mathrm{unvec}(\mathbf{w}_{t+1})
$$

---

#### 2. Применение GCN с обновлёнными весами

Обновлённая матрица $W_{t+1}$ используется в обычной операции GCN:

$$
H_{t+1} = \sigma(\hat{A}_t H_t W_{t+1})
$$

где  
- $H_t$ — признаки узлов на шаге $t$,  
- $\hat{A}_t$ — нормализованная матрица смежности snapshot-а $G_t$,  
- $\sigma$ — нелинейность (ReLU).

---

###  Полная рекурсия

GCN-переход:

$$
H_{t+1} = \mathrm{ReLU}(\hat{A}_t H_t W_t)
$$

Эволюция весов:

$$
W_{t+1} = \mathrm{GRUCell}(\mathrm{vec}(W_t), \mathrm{vec}(W_t))
$$

Так EvolveGCN-O отслеживает структуру графа во времени.

---

###  Архитектурные компоненты

#### **1. Node features**
Если у графа нет признаков (как у Wikipedia):

$$
H_0 = X \in \mathbb{R}^{N \times d}
$$

Мы используем обучаемую матрицу embedding-ов узлов.



#### **2. GCNConv**
GCN использует матрицу весов $W_t$, обновлённую через GRU.



#### **3. GRUCell**
GRU принимает векторизированную матрицу весов:

$$
\mathbf{w}_{t+1} = \mathrm{GRUCell}(\mathbf{w}_t, \mathbf{w}_t)
$$

Это гарантирует устойчивое обновление $W_t$.



In [ ]:
class EvolveGCN_O_Layer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim

        # TODO

    # Функция обновления весов W (W_t → W_{t+1})
    def evolve_weights(self):
        # TODO

    # Прямой проход слоя EvolveGCN-O
    def forward(self, x, edge_index):
        # TODO
        return self.gcn(x, edge_index).relu()



# Обёртка для одного слоя EvolveGCN-O
class EvolveGCN_O(nn.Module):
    def __init__(self, in_dim, hid_dim):
        super().__init__()
        # Обычно архитектура имеет один слой O или стек слоёв.
        self.layer = EvolveGCN_O_Layer(in_dim, hid_dim)

    def forward(self, x, edge_index):
        return self.layer(x, edge_index)



In [ ]:
# reindex users and items into a single node space
users = df.user.unique()
items = df.item.unique()

u_map = {u: i for i, u in enumerate(users)}
i_map = {it: i + len(users) for i, it in enumerate(items)}

df["src"] = df.user.map(u_map)
df["dst"] = df.item.map(i_map)

num_users = len(users)
num_items = len(items)
num_nodes = num_users + num_items

df = df[["src", "dst", "timestamp"]]

In [ ]:
# Размерность эмбеддинга узлов.
dim = 64

# Создаём обучаемую матрицу признаков узлов (node embeddings).
# В отличие от статического baseline'a с X = I (one-hot),
# здесь мы задаём X как TRAINABLE embedding.
x = nn.Embedding(num_nodes, dim)

# Инициализация признаков узлов.
nn.init.xavier_uniform_(x.weight)


### Training Loop for EvolveGCN-O

На каждой эпохе:

1. Берём текущие признаки узлов `h_t`
2. Для каждого snapshot-а графа:
   - обновляем веса GCN через GRU (`W_t → W_{t+1}`)
   - применяем GCN к признакам `h_t`
   - вычисляем loss на позитивных и негативных рёбрах
   - обновляем параметры модели
   - `detach()` скрытого состояния (нет BPTT)
3. На валидации прогоняем узлы через все train-snapshots
4. Считаем AUC по real/fake рёбрам



In [ ]:
model = EvolveGCN_O(dim, dim)

opt = torch.optim.Adam(list(model.parameters()) + list(x.parameters()), lr=1e-3)

for epoch in range(3):
    # 1. Начальное состояние узлов (node features)
    h_t = x.weight


    # 2. Проход по всем snapshot-графам (по временным шагам)
    for snap in train_s:

        # Строим edge_index для snapshot-а
        ei = to_ei(snap)

        # Прямой проход EvolveGCN:
        #
        #   1) evolve_weights() — GRU обновляет W_t → W_{t+1}
        #   2) GCNConv применяет новую матрицу W_{t+1}
        #
        # h_next — новые скрытые состояния узлов после применения
        #          GCN к snapshot-у G_t.
        h_next = model(h_t, ei)


        # 3. Позитивные и негативные рёбра в snapshot-е
        pos = torch.tensor(snap[["src","dst"]].values)
        neg = neg_sample(snap, num_users, num_items)


        # 4. Вычисление скорингов для всех рёбер
        # Позитивные примеры: реальные взаимодействия
        pos_scores = torch.stack([score(u, v, h_next) for u, v in pos])

        # Негативные примеры: случайные пары user-item
        neg_scores = torch.stack([score(u, v, h_next) for u, v in neg])


        # 5. Функция потерь (binary cross-entropy)
        #   L = - ( log σ(pos) + log (1 - σ(neg)) )
        #
        # Мы хотим, чтобы:
        #   real edges   → score высокий
        #   fake edges   → score низкий
        loss = -(torch.log(torch.sigmoid(pos_scores)).mean() +
                 torch.log(1 - torch.sigmoid(neg_scores)).mean())


        # 6. Backprop + Optimization
        opt.zero_grad()
        loss.backward()
        opt.step()

        # 7. Детач скрытого состояния между snapshots
        h_t = h_next.detach()


    # 8. Валидация (без градиентов)
    h_val = x.weight.clone()

    # Прогоняем валидаторные узловые признаки через ВСЕ train-snapshots,
    # чтобы получить окончательные embedding для времени T
    for snap in train_s:
        h_val = model(h_val, to_ei(snap)).detach()

    # Формируем позитивные и негативные пары
    pos = torch.tensor(val_df[["src","dst"]].values)
    neg = neg_sample(val_df, num_users, num_items)

    pos_s = torch.stack([score(u, v, h_val) for u, v in pos])
    neg_s = torch.stack([score(u, v, h_val) for u, v in neg])

    y = np.concatenate([np.ones(len(pos_s)), np.zeros(len(neg_s))])
    s = torch.cat([pos_s, neg_s]).detach().numpy()

    # AUC — качество различения real vs fake рёбер
    auc = roc_auc_score(y, s)

    print(f"[EvolveGCN-O] epoch={epoch}, val AUC={auc:.4f}")


###  Интуитивная интерпретация

- Граф меняется во времени  
- Значит и оптимальные GCN-веса должны меняться  
- GRU учит динамику изменения структуры графа,  
  обновляя веса GCN вместо признаков узлов

Итог:

> Модель адаптируется к структуре графа,  
> "забывая" старые паттерны и "учась" новым.

---

###  Когда EvolveGCN-O полезен?

- когда у узлов есть признаки  
- когда структура графа меняется постепенно  
- когда snapshots достаточно плотные  
- когда важно моделировать динамику параметров, а не динамику эмбеддингов

---

###  Когда EvolveGCN-O слабее?

- на event-based графах (Wikipedia, Reddit, MOOC)  
- на двудольных user–item графах  
- при отсутствии node features  
- при слишком разреженных snapshot-графах  

В этих случаях лучше работают:

- TGAT  
- TGN  
- DyRep  
- JODIE



## TGAT — Temporal Graph Attention Network



TGAT (Xu et al., ICLR 2020) — это event-based динамическая GNN, которая обрабатывает граф как поток событий  
$(u, v, t)$ в непрерывном времени.  
Главная идея TGAT — объединить:

- временное кодирование (time encoding),
- attention-механизм,
- соседство только по историческим событиям (temporal neighborhood),
- агрегирование информации строго в графе событий, а не в snapshot’ах.

TGAT — одна из самых мощных моделей для временных графов (Wikipedia, Reddit, MOOC, UCI, BitcoinOTC).



### Основная идея TGAT

Вместо того чтобы использовать фиксированные snapshot-графы, TGAT работает в режиме:

$$
(u, v, t_1),\; (u_2, v_3, t_2),\; (u_5, v_1, t_3),\; \dots
$$

То есть модель наблюдает события в хронологическом порядке и для каждого события формирует embedding узлов,  
учитывая:

- историю их прошлых взаимодействий,
- временной интервал между событиями,
- структуру локального графа.

---

### Компоненты TGAT

TGAT состоит из трёх ключевых частей:

1. **Time Encoding** — переводит временные интервалы в векторы  
2. **Temporal Multi-Head Attention** — выбирает, какие соседи важнее  
3. **Aggregating historical neighbors** — собирает информацию от прошлых событий  

---

### Time Encoding (TE)

Для узла $u$ в момент времени $t$ TGAT вычисляет time encoding, который отражает положение события во времени.

TGAT использует синусоидальные функции:

$$
\text{TE}(t) = \left[ \sin(\omega_1 t), \cos(\omega_1 t), \dots, \sin(\omega_k t), \cos(\omega_k t) \right]
$$

Это позволяет:

- кодировать время как непрерывную величину,
- получать периодические и масштабируемые представления,
- эффективно сравнивать свежие и старые события.

TE(t) всегда имеет фиксированную размерность, например 64.

---

### Входы TGAT

Для узла $u$ TGAT формирует входной вектор:

$$
z_u = [h_u \,\, \Vert \,\, \text{TE}(t)]
$$

где  
- $h_u$ — learnable embedding узла  
- $TE(t)$ — временное кодирование  

Размерность:

$$
\dim(z_u) = d_{\text{emb}} + d_{\text{time}}
$$

---

### Temporal Attention

TGAT использует attention, как в Transformer, но с учётом времени.

У каждого события $(u, v, t)$ есть:

- запрос $Q$ (Query — кого обновляем)
- ключ $K$ (Key — как решаем, кто важен)
- значение $V$ (Value — что именно сосед передаёт)

определённые как:

$$
Q_u = W_Q z_u, \qquad
K_v = W_K z_v, \qquad
V_v = W_V z_v
$$

Тогда attention-веса:

$$
\alpha_{uv} = \frac{Q_u \cdot K_v}{\sqrt{d_k}}
$$

После softmax:

$$
a_{uv} = \text{softmax}(\alpha_{uv})
$$

Финальное сообщение:

$$
m_u = \sum_{v \in \mathcal{N}(u)}
    a_{uv} \, V_v
$$

И итоговый embedding:

$$
h_u' = W_O m_u
$$

Это аналог self-attention, но внимание рассчитывается между узлом и его историческими соседями.

---

### Temporal Neighborhood

Главное отличие от обычного GAT:

TGAT использует только прошлые события узла:

$$
\mathcal{N}(u, t) = \{ v \mid (u, v, t'),\; t' < t \}
$$

Это позволяет:

- строить правильную временную причинность,
- обучать модель предсказывать будущее из истории,
- избегать information leakage.

---

### Механизм TGAT в одном шаге

Пусть пришло событие $(u, v, t)$.

TGAT делает:

1. формирует $z_u = [h_u \Vert TE(t)]$  
2. собирает прошлых соседей $v_1, v_2, \dots$  
3. формирует набор $z_{v_i}$  
4. вычисляет attention:  
   $a_{uv_i}$  
5. агрегирует значение $V_{v_i}$
6. обновляет embedding узла $u$

---

### Финальная формула TGAT обновления

$$
h_u(t) =
W_O \left(
\sum_{v \in \mathcal{N}(u,t)}
    \text{softmax}\left(
        \frac{(W_Q z_u) \cdot (W_K z_v)}{\sqrt{d_k}}
    \right)
    (W_V z_v)
\right)
$$

---



### Reindexing user and item IDs into a single node space

User–item графы являются двудольными: пользователи и объекты находятся в разных множествах.  
Но любая GNN (GCN, EvolveGCN, TGAT, GraphSAGE) требует, чтобы все узлы имели *общую нумерацию*  
в диапазоне `[0 .. num_nodes-1]`.

Поэтому мы:

1. объединяем user-id и item-id в единый список узлов,  
2. создаём словарь `mapping` для перенумерации узлов,  
3. применяем этот словарь к столбцам `df.user` и `df.item`.

Теперь каждый пользователь и каждый объект имеет собственный **целочисленный индекс узла**.


In [ ]:
url = "https://snap.stanford.edu/jodie/wikipedia.csv"
df = pd.read_csv(url, header=None, usecols=[0,1,2])
df.columns = ["user","item","timestamp"]

# clean timestamps
df["timestamp"] = pd.to_numeric(df["timestamp"].astype(str).str.strip(), errors="coerce")
df = df.dropna(subset=["timestamp"])
df["timestamp"] = df["timestamp"].astype(int)

df = df.sort_values("timestamp").reset_index(drop=True)

print(df.shape)
df.head()


(157474, 3)


/tmp/ipython-input-728066216.py:2: DtypeWarning: Columns (0,1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url, header=None, usecols=[0,1,2])


,user,item,timestamp
0,0,0,0
1,1,1,36
2,1,1,77
3,2,2,131
4,1,1,150


In [ ]:
# Объединяем user-идентификаторы и item-идентификаторы
nodes = pd.concat([
    df.user.astype(str),
    df.item.astype(str)
]).unique()

# Создаём отображение "старый id → новый индекс"
mapping = {x: i for i, x in enumerate(nodes)}

# Общее число узлов = число уникальных пользователей + число уникальных items
num_nodes = len(mapping)


# Применяем отображение к user и item
# Преобразование делаем через .map(mapping).
df.user = df.user.astype(str).map(mapping)
df.item = df.item.astype(str).map(mapping)


In [ ]:
train_size = int(0.8 * len(df))
train_df = df.iloc[:train_size].copy()
val_df = df.iloc[train_size:].copy()

# ---------- НОВОЕ: нормализация таймстемпов ----------
t_min = df.timestamp.min()
t_max = df.timestamp.max()

train_df["timestamp"] = (train_df.timestamp - t_min) / (t_max - t_min + 1e-9)
val_df["timestamp"]   = (val_df.timestamp - t_min) / (t_max - t_min + 1e-9)

# ---------- Преобразуем в tensor ----------
s_train = torch.tensor(train_df.user.values)
d_train = torch.tensor(train_df.item.values)
t_train = torch.tensor(train_df.timestamp.values, dtype=torch.float)

s_val = torch.tensor(val_df.user.values)
d_val = torch.tensor(val_df.item.values)
t_val = torch.tensor(val_df.timestamp.values, dtype=torch.float)


In [ ]:
def neg_sample(n, num_nodes):
    return torch.randint(0, num_nodes, (n,))


### Full TGAT Model

In [ ]:
# Time Encoding
class TimeEncode(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        # TODO

    def forward(self, t):
        # TODO

        return out



# Temporal Attention
class TemporalAttention(nn.Module):
    def __init__(self, embed_dim=64, time_dim=64, heads=2):
        super().__init__()
        # TODO

    def forward(self, q_feat, k_feat, q_time, k_time):
        # TODO
        return self.WO(out)


# TGAT Encoder
class TGATEncoder(nn.Module):
    def __init__(self, num_nodes, dim=64, time_dim=64):
        super().__init__()

        # TODO

    def forward(self, src, dst, t):
        # TODO

        return out_src, out_dst



# Link Predictor
class LinkPredictor(nn.Module):
    def forward(self, h1, h2):
        # Скалярное произведение эмбеддингов:
        # высокий score → предполагается, что ребро существует
        return (h1 * h2).sum(dim=1)



# Full TGAT LP Model
class TGATLP(nn.Module):
    def __init__(self, num_nodes):
        super().__init__()
        # Полный TGAT-энкодер:
        #   • node embeddings
        #   • time encoding
        #   • temporal attention
        self.enc = TGATEncoder(num_nodes, dim=64, time_dim=64)

    def forward(self, s, d, t):
        # Вычисляем эмбеддинги для источника и приёмника
        out_src, out_dst = self.enc(s, d, t)

        # Считаем score через скалярное произведение
        return (out_src * out_dst).sum(dim=1)



In [ ]:
def eval_tgat(model):
    # 1. Извлекаем валидационные события:
    #    • s — источники (users)
    #    • d — приёмники (items)
    #    • t — timestamp события
    #
    # Это реальные взаимодействия (positive edges).
    s = torch.tensor(val_df.user.values)
    d = torch.tensor(val_df.item.values)
    t = torch.tensor(val_df.timestamp.values, dtype=torch.float)

    # 2. Получаем скоры модели TGAT для всех позитивных рёбер:
    #
    # model(s, d, t) возвращает:
    #     score(u_i, v_i, t_i)
    #
    # где score — скалярное произведение выходов TGATEncoder.
    pos = model(s, d, t)

    # 3. Генерируем отрицательные примеры (negative edges)
    # nd — случайные item-узлы того же размера, что и s.
    # neg_sample(num_pos, num_nodes) генерирует пары (u, v_fake),
    # которых нет в графе.
    nd = neg_sample(len(s), num_nodes)

    # 4. Скоры TGAT на отрицательных рёбрах
    neg = model(s, nd, t)

    # 5. Формируем разметку (labels)
    #
    #   1 → positive edges
    #   0 → negative edges
    #
    # Длины pos и neg одинаковые.
    y = torch.cat([
        torch.ones(len(pos)),
        torch.zeros(len(neg))
    ]).numpy()


    # 6. Собираем предсказанные скоры:
    scores = torch.cat([pos, neg]).detach().numpy()

    # 7. AUC-ROC:
    return roc_auc_score(y, scores)


### Train TGAT

In [ ]:
model = TGATLP(num_nodes)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)


s_train = torch.tensor(train_df.user.values)
d_train = torch.tensor(train_df.item.values)
t_train = torch.tensor(train_df.timestamp.values, dtype=torch.float)


### Обучение TGAT на задаче link prediction

На каждой эпохе:

1. **Позитивные события:**  
   TGAT получает реальные взаимодействия `(u, v, t)` и вычисляет score.

2. **Негативные события:**  
   Для каждого real-события генерируется отрицательный пример `(u, fake_v, t)`.

3. **Loss:**  
   Используется бинарная кросс-энтропия:  
   реальные события должны иметь высокую вероятность,  
   негативные — низкую.

4. **Обновление параметров:**  
   Оптимизатор Adam обновляет:
   - embedding узлов,  
   - параметры time-encoder,  
   - Q/K/V матрицы attention,  
   - выходной проектор WO.

5. **AUC:**  
   AUC-ROC измеряет, насколько хорошо TGAT различает real vs fake ссылки.


In [ ]:
for ep in range(30):
    # 1. Позитивные примеры (реальные события)
    pos = model(s_train, d_train, t_train)

    # 2. Генерация негативных событий
    nd  = neg_sample(len(s_train), num_nodes)
    neg = model(s_train, nd, t_train)

    # 3. Функция потерь (binary cross-entropy)
    loss = -(torch.log(torch.sigmoid(pos)).mean() +
             torch.log(1 - torch.sigmoid(neg)).mean())

    # 4. Backpropagation + Update
    opt.zero_grad()
    loss.backward()
    opt.step()

    # 5. AUC на валидации
    # eval_tgat(model) вызывает модель на val_df
    # и считает AUC по "real vs fake".
    auc = eval_tgat(model)

    print(f"Epoch {ep}: loss={loss.item():.4f}, AUC={auc:.4f}")


### Зачем нужен TGAT?

TGAT идеально подходит для графов, где:

- события поступают в реальном времени,  
- связь зависит от прошлого,  
- важно учитывать "recency" событий,  
- граф очень разрежен (Wikipedia, Reddit, MOOC).

TGAT **стабильнее**, чем snapshot-модели (EvolveGCN),  
и **мощнее**, чем GCN/GAT на статичных графах.

---

### Особенности TGAT

- **Непрерывное время** (continuous-time)  
- **Attention** вместо усреднения соседей  
- **Time Encoding** вместо snapshot ID  
- **Event-based** архитектура  
- **Работает на двудольных графах** (user-item), где EvolveGCN проваливается  
- **State-of-the-art** на многолетних задачах предсказания взаимодействий  

---

### Итог

TGAT вычисляет embedding узла в момент времени $t$ как  
attention-взвешенную сумму сообщений от **прошлых** событий,  
где каждое сообщение усиливается или ослабляется **разницей во времени**  
и **релевантностью соседа**, вычисленной через attention.

Это делает TGAT одной из самых мощных моделей для динамического графового анализа.
